In [ ]:
from IPython.display import HTML
display(HTML("<style>.rendered_html { font-size: 1.3em; } .code_cell .input_area { font-size: 1.1em; }</style>"))

# 5.1 Course Bottleneck Detection with Unsupervised Learning
- No target variable — we **discover** structure
- K-Means clustering + PCA visualization
- Applied to a real institutional question: which courses are bottlenecks?
- Uses matplotlib/seaborn for static visualizations

## Supervised vs. Unsupervised
- **Supervised** (Modules 2-4): Predict a known target (DEPARTED)
- **Unsupervised** (this module): Find hidden groupings without a target
- Key question: "What natural patterns exist in our data?"

## The Provost's Question
> "We have data on 1,000 course sections. Some act as **bottlenecks** — high DFW rates,
> low pass rates, repeats. Can we **discover** clusters of problematic courses and rank
> them for curriculum reform?"

## Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 12

print("Libraries loaded!")

## Load the Course Section Data
The synthetic course data is pre-built and saved to `bottleneck.csv` in the Course 3
`data/` folder, so every run analyzes the same 1,000 sections.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
project_path = '/content/drive/MyDrive/Applied-Data-Analytics-For-Higher-Education-Course-3'
data_filepath = '/data/'

course_df = pd.read_csv(f'{project_path}{data_filepath}bottleneck.csv')

print(f"Dataset: {course_df.shape[0]:,} course sections x {course_df.shape[1]} variables")
course_df.head()

## Step 1: Scale the Data
- K-Means uses Euclidean distance
- Without scaling, variables with larger ranges (ENROLLMENT) dominate small ones (DFW_RATE)
- StandardScaler: mean=0, std=1

In [ ]:
scaler = StandardScaler()
course_scaled = scaler.fit_transform(course_df)

print("Before scaling:")
print(f"  ENROLLMENT range: {course_df['ENROLLMENT'].min()} - {course_df['ENROLLMENT'].max()}")
print(f"  DFW_RATE range:   {course_df['DFW_RATE'].min():.2f} - {course_df['DFW_RATE'].max():.2f}")
print(f"\nAfter scaling (mean ~ 0, std ~ 1):")
print(f"  ENROLLMENT range: {course_scaled[:, 0].min():.2f} - {course_scaled[:, 0].max():.2f}")
print(f"  DFW_RATE range:   {course_scaled[:, 1].min():.2f} - {course_scaled[:, 1].max():.2f}")

## Step 2: Choosing the Number of Clusters
- **Silhouette Score**: how compact/separated the clusters are (higher = better)
- We sweep k = 2..10 — but the metric alone does **not** pick a single k for us
- We choose **k = 4**, and the cells right after the plot show why that beats the
  silhouette-maximizing choice for *this* decision

In [ ]:
sils_c = []
for k in range(2, 11):
    km = KMeans(n_clusters=k, n_init=10, random_state=RANDOM_STATE)
    sils_c.append(silhouette_score(course_scaled, km.fit_predict(course_scaled)))

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(range(2, 11), sils_c, 'go-', linewidth=2, markersize=8)
ax.set_xlabel('Number of Clusters (k)'); ax.set_ylabel('Silhouette Score')
ax.set_title('Silhouette Analysis - Course Sections')
ax.axvline(x=4, color='red', linestyle='--', alpha=0.7, label='k = 4 (chosen)')
ax.legend(); plt.tight_layout(); plt.show()

### Why k = 4 when the silhouette peaks at k = 3?

The silhouette score is **highest at k = 3** (≈ 0.36) and *falls* to ≈ 0.26 at k = 4.
Read literally, the metric says "use 3." We pick 4 anyway — here's the honest reasoning:

1. **The bottleneck cluster is identical at k = 3 and k = 4** — the same ~200 high-DFW,
   low-pass, high-repeat sections. **Our answer to the Provost does not depend on k.**
2. **Going 3 → 4 splits the large healthy bucket into two real course types** (small
   high-prerequisite upper-division vs. mid-size core), not noise — a distinction a dean
   would act on.
3. **The silhouette dip is expected**: splitting one blob into two adjacent-but-real
   sub-groups always lowers the score a little. A mild drop is the price of a more
   actionable segmentation.

**Rule:** silhouette *narrows the field* (here to 3–4); **interpretability makes the call.**
Let's verify claims (1) and (2).

In [ ]:
# Compare k=3 vs k=4 to justify the choice
compare_cols = ['ENROLLMENT', 'DFW_RATE', 'PASS_RATE', 'REPEAT_RATE',
                'PCT_FIRST_YEAR', 'IS_GATEWAY', 'PREREQUISITE_COUNT']

for k in (3, 4):
    km_tmp = KMeans(n_clusters=k, n_init=10, random_state=RANDOM_STATE)
    labels_tmp = km_tmp.fit_predict(course_scaled)
    prof_tmp = course_df[compare_cols].groupby(labels_tmp).mean().round(2)
    prof_tmp['n'] = pd.Series(labels_tmp).value_counts().sort_index().values
    print(f"k = {k}  |  silhouette = {silhouette_score(course_scaled, labels_tmp):.3f}")
    print(prof_tmp.to_string()); print()

print("The bottleneck row (~200 sections) is identical in both; k=4 just splits the")
print("healthy bucket into upper-division vs. core. That extra split is why we pick k=4.")

## Step 3: Fit K-Means with k=4

In [ ]:
km_course = KMeans(n_clusters=4, n_init=10, random_state=RANDOM_STATE)
course_df['Cluster'] = km_course.fit_predict(course_scaled)

print(f"Silhouette Score: {silhouette_score(course_scaled, course_df['Cluster']):.3f}")
print(f"\nCluster Sizes:")
print(course_df['Cluster'].value_counts().sort_index())

## Step 4: PCA for Visualization
- 13 variables -> can't plot directly
- PCA compresses to 2 dimensions that capture the most variance

In [ ]:
pca_c = PCA(n_components=2, random_state=RANDOM_STATE)
course_pca = pca_c.fit_transform(course_scaled)

fig, ax = plt.subplots(figsize=(10, 7))
scatter = ax.scatter(course_pca[:, 0], course_pca[:, 1],
    c=course_df['Cluster'], cmap='Set2', alpha=0.6, s=30, edgecolors='w', linewidth=0.3)
ax.set_xlabel(f'PC1 ({pca_c.explained_variance_ratio_[0]:.1%} variance)')
ax.set_ylabel(f'PC2 ({pca_c.explained_variance_ratio_[1]:.1%} variance)')
ax.set_title('Course Section Segments (PCA Projection)')
plt.colorbar(scatter, label='Cluster')
plt.tight_layout(); plt.show()

print(f"PC1 + PC2 explain {sum(pca_c.explained_variance_ratio_[:2]):.1%} of variance")

## Step 5: Cluster Profiling
- Most important step for institutional use
- Compute mean of each variable per cluster
- The **bottleneck cluster** is red on DFW_RATE / REPEAT_RATE and low on PASS_RATE

In [ ]:
profile_c = course_df.groupby('Cluster').mean().round(3)
profile_c['Count'] = course_df['Cluster'].value_counts().sort_index()
print("=" * 100)
print("CLUSTER PROFILES - Course Bottleneck Detection")
print("=" * 100)
print(profile_c.to_string())

profile_cz = (profile_c.drop(columns='Count') - profile_c.drop(columns='Count').mean()) / profile_c.drop(columns='Count').std()
fig, ax = plt.subplots(figsize=(14, 5))
sns.heatmap(profile_cz, annot=True, fmt='.2f', cmap='RdYlGn_r', center=0, linewidths=1, ax=ax)
ax.set_title('Course Cluster Profiles (z-scored - red = concerning)')
ax.set_ylabel('Cluster')
plt.tight_layout(); plt.show()

## Step 6: Risk Prioritization Index
Combine the metrics that matter into one ranked score so leadership knows which
specific sections to review first:
- DFW Rate (0.4) + Repeat Rate (0.3) + Inverse Pass Rate (0.3)

In [ ]:
course_df['RISK_INDEX'] = (
    0.4 * (course_df['DFW_RATE'] / course_df['DFW_RATE'].max()) +
    0.3 * (course_df['REPEAT_RATE'] / course_df['REPEAT_RATE'].max()) +
    0.3 * ((1 - course_df['PASS_RATE']) / (1 - course_df['PASS_RATE']).max())
).round(3)

top_risk = course_df.nlargest(20, 'RISK_INDEX')[
    ['ENROLLMENT', 'DFW_RATE', 'PASS_RATE', 'REPEAT_RATE', 'IS_GATEWAY', 'RISK_INDEX', 'Cluster']
]

print("=" * 80)
print("TOP 20 HIGHEST-RISK COURSE SECTIONS")
print("=" * 80)
print(top_risk.to_string())

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
course_df.boxplot(column='RISK_INDEX', by='Cluster', ax=axes[0])
axes[0].set_title('Risk Index by Cluster')
axes[0].set_xlabel('Cluster'); axes[0].set_ylabel('Risk Index')

for c in sorted(course_df['Cluster'].unique()):
    axes[1].hist(course_df[course_df['Cluster'] == c]['RISK_INDEX'],
                 alpha=0.5, bins=20, label=f'Cluster {c}')
axes[1].set_xlabel('Risk Index'); axes[1].set_ylabel('Count')
axes[1].set_title('Risk Index Distribution by Cluster')
axes[1].legend()
plt.tight_layout(); plt.show()

## The Unsupervised Learning Pipeline
```
1. Scale data (StandardScaler)
2. Find optimal k (Silhouette)
3. Fit K-Means
4. Visualize with PCA
5. Profile clusters (means + heatmap)
6. Rank with a Risk Prioritization Index
```

## Key Takeaways
- Unsupervised learning discovers hidden structure without a target
- Always scale before K-Means
- Use the Silhouette score together with interpretability to pick k
- PCA for visualization
- Cluster profiling + a Risk Index are the deliverables leadership acts on
- **Module 5 Complete! Next: Module 6 (Special Topics - Optional)**